# Notebook 1: EXPLORING AND CLEANING OLIST E-COMMERCE DATA
-------------------------------------------------------------------------------------------------

### Author: Emmanuel Aregbesola
### Date: April, 2026



In [1]:
# import pandas library

import pandas as pd

In [2]:
# Load the data

files = {
    'customers' : 'olist_customers_dataset.csv',
    'geolocation' : 'olist_geolocation_dataset.csv',
    'order_items' : 'olist_order_items_dataset.csv',
    'payment' : 'olist_order_payments_dataset.csv',
    'reviews' : 'olist_order_reviews_dataset.csv',
    'orders' : 'olist_orders_dataset.csv',
    'products' : 'olist_products_dataset.csv',
    'sellers' : 'olist_sellers_dataset.csv',
    'product_category' : 'product_category_name_translation.csv'
}

data = {}

for name, file in files.items():
    data[name] = pd.read_csv(file)
    
    
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [3]:
# create placeholders for all dataframe

customers_df = data['customers']
geolocation_df = data['geolocation']
order_items_df = data['order_items']
payments_df = data['payment']
reviews_df = data['reviews']
orders_df = data['orders']
products_df = data['products']
sellers_df = data['sellers']
product_category_df = data['product_category']

In [4]:
# basic exploration function 
def basic_exploration(df):
    # 1. Visual Header
    print(f"{'='*30}")
    print("      VITAL STATISTICS")
    print(f"{'='*30}")
    
    # 2. Dimensions and Red Flags
    print(f"Shape: {df.shape}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print("-" * 30)
    
    # 3. Missing Data Profile (Count and Percentage)
    print("MISSING DATA PROFILE:")
    null_count = df.isna().sum()
    null_percent = (null_count / len(df)) * 100
    null_table = pd.concat([null_count, null_percent.round(2)], axis=1)
    null_table.columns = ['Count', 'Percentage (%)']
    print(null_table)
    print("-" * 30)
    
    # 4. Data Types (Clean View)
    print("DATA TYPES:")
    print(df.dtypes)
    print("-" * 30)
    
    # 5. Summary Statistics (Numeric & Categorical)
    print("SUMMARY STATISTICS:")
    # include='all' ensures we see unique counts for strings, not just math for numbers
    display(df.describe(include='all')) 
    print(f"{'='*30}\n")

## Data Cleaning

This section involvces taking each tables individually and cleaning them.

## Payments Dataframe

### order_id: unique identifier of an order.
### payment_sequential: a customer may pay an order with more than one payment method. If he does so, a sequence will be created to
### payment_type: method of payment chosen by the customer.
### payment_installments: number of installments chosen by the customer.
### payment_value: transaction value.

In [5]:
basic_exploration(payments_df)

      VITAL STATISTICS
Shape: (103886, 5)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                      Count  Percentage (%)
order_id                  0             0.0
payment_sequential        0             0.0
payment_type              0             0.0
payment_installments      0             0.0
payment_value             0             0.0
------------------------------
DATA TYPES:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


In [6]:
# check basic overview of payments dataframe

payments_df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [7]:
# Check for frequency for payment type

payments_df['payment_type'].value_counts()

credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: payment_type, dtype: int64

In [8]:
# create a filter dataframe for the not_defined value in the payment_type column

payments_mask = payments_df['payment_type'] == 'not_defined'

payments_df.loc[payments_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [9]:
# check if the order_id for the not_defined values appears in the entire dataframe

problem_id = payments_df.loc[payments_mask, 'order_id']
check_id = payments_df[payments_df['order_id'].isin(problem_id)]
check_id

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [10]:
# drop the not_defined rows

payments_df = payments_df[~payments_mask].reset_index(drop=True)

In [11]:
# validate the drop

payments_df['payment_type'].value_counts()

credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
Name: payment_type, dtype: int64

In [12]:
# filter the dataframe to condition of payment_value = 0

payment_zero_mask = payments_df['payment_value'] == 0.0

payments_df.loc[payment_zero_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
62672,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77883,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
100763,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [13]:
# filter dataframe to condition of payment_installments = 0

installment_zero_mask = payments_df['payment_installments'] == 0

payments_df.loc[installment_zero_mask]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79012,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [14]:
# Fix logical error

payments_df.loc[payments_df['payment_installments'] == 0, 'payment_installments'] = 1

In [15]:
# Validate the payment column

basic_exploration(payments_df)

      VITAL STATISTICS
Shape: (103883, 5)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                      Count  Percentage (%)
order_id                  0             0.0
payment_sequential        0             0.0
payment_type              0             0.0
payment_installments      0             0.0
payment_value             0             0.0
------------------------------
DATA TYPES:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object
------------------------------
SUMMARY STATISTICS:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103883,103883.000000,103883,103883.000000,103883.000000
unique,99437,NaN,4,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092681,NaN,2.853422,154.104831
std,NaN,0.706594,NaN,2.687054,217.495628
min,NaN,1.000000,NaN,1.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.800000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.840000


### Summary on payments cleaning

1. After running the basic exploration function on the payments dataframe, it was found that there are no null values, duplicates and the data types are correct. The summary statistics didnt reveal anything out of place except for the fact that there were zero values in the `payment_value`  and `payment_installments` column.            

                   
2. For validity purposes, a value_counts was ran to check what types of payment existed, there it was found that there were `payment_types` tagged `not_defined`. After filtering, it was found that the `not_defined` rows were a sort of ghost records. This was after order_ids that had the `not_defined` values were checked across the dataframe, if there could be another record. They had zero value, and only one payment sequential which led to the rows being dropped.    
Note: These `not_defined` rows were only 3.              

                           
3. We also explored the rows that had zero values. It was found that the `payment_value` that had zero were transactions that were paid with vouchers. Since these are legit transactions, they were not dropped. In the `payment_installments` column, it was concluded that the zer values were logical errors. This is because it is highly unlikely for a transaction with zero installments to have a price. It was decided that the zero values were changed to `1` since the mode of the column is that said value.


                                     
**Rows before cleaning** = 103886     
**Rows before cleaning** = 103883       
**Rows dropped** = 3 

## Orders Dataframe

### order_id: unique identifier of the order.
### customer_id: key to the customer dataset. Each order has a unique customer_id.
### order_status: Reference to the order status (delivered, shipped, etc).
### order_purchase_timestamp: Shows the purchase timestamp.
### order_approved_at: Shows the payment approval timestamp.
### order_delivered_carrier_date: Shows the order posting timestamp. When it was handled to the logistic partner.
### order_delivered_customer_date: Shows the actual order delivery date to the customer.
### order_estimated_delivery_date: Shows the estimated delivery date that was informed to customer at the purchase moment.

In [16]:
# explore the orders dataframe

basic_exploration(orders_df)

      VITAL STATISTICS
Shape: (99441, 8)
Duplicate Rows: 0
------------------------------
MISSING DATA PROFILE:
                               Count  Percentage (%)
order_id                           0            0.00
customer_id                        0            0.00
order_status                       0            0.00
order_purchase_timestamp           0            0.00
order_approved_at                160            0.16
order_delivered_carrier_date    1783            1.79
order_delivered_customer_date   2965            2.98
order_estimated_delivery_date      0            0.00
------------------------------
DATA TYPES:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
------------------------------
SUM

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [17]:
# convert date columns to datetime

date_columns = ['order_purchase_timestamp',
                'order_approved_at',
                'order_delivered_carrier_date',
                'order_delivered_customer_date',
                'order_estimated_delivery_date'
]

for col in date_columns:
    orders_df[col] = pd.to_datetime(orders_df[col], errors='coerce')

In [18]:
# validate the datetime change

orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [19]:
# checking the value count of order_status in approved transactions

orders_df[orders_df['order_approved_at'].isna()]['order_status'].value_counts()

canceled     141
delivered     14
created        5
Name: order_status, dtype: int64

In [20]:
# checking the dataframe for approved transaction that are delivered

approved_delivered = (orders_df['order_status'] == 'delivered') & (orders_df['order_approved_at'].isna())

orders_df.loc[approved_delivered]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


In [21]:
# filling the approved_delivered mask with the purchase_timestamp

orders_df.loc[approved_delivered, 'order_approved_at'] = orders_df.loc[approved_delivered, 'order_purchase_timestamp']

In [22]:
# confirm the fill

approved_delivered = (orders_df['order_status'] == 'delivered') & (orders_df['order_approved_at'].isna())
orders_df[approved_delivered].shape

(0, 8)

In [26]:
# Check delivered orders that have no delievered timestamp
delivered_no_customer = (orders_df['order_status'] == 'delivered') & (orders_df['order_delivered_customer_date'].isna())

orders_df.loc[delivered_no_customer]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [29]:
# check the shape of the mask

orders_df.loc[delivered_no_customer].shape

(8, 8)

In [28]:
# drop the orders that was delivered but have no delivery timestamp

orders_df = orders_df[~delivered_no_customer].reset_index(drop=True)

In [30]:
# check the statuses in order_status

orders_df['order_status'].value_counts()

delivered      96470
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_status, dtype: int64

In [31]:
# Check for canceled orders that has delivered timestamp

canceled_integrity = (orders_df['order_status'] == 'canceled') & (orders_df['order_delivered_customer_date'].notna())

orders_df.loc[canceled_integrity]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8790,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58263,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59329,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92631,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94393,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


In [32]:
# check the shape

orders_df.loc[canceled_integrity].shape

(6, 8)

In [33]:
# drop the canceled integrity table

orders_df = orders_df[~canceled_integrity].reset_index(drop=True)

In [38]:
# check if approved order has a approved date
order_approved = (orders_df['order_status'] == 'approved') & (orders_df['order_approved_at'].isna())

orders_df.loc[order_approved]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [39]:
orders_df.loc[order_approved].shape

(0, 8)

In [40]:
order_created = (orders_df['order_status'] == 'created') & (orders_df['order_purchase_timestamp'].isna())

orders_df.loc[order_created]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [41]:
orders_df.loc[order_created].shape

(0, 8)